# Notebook 18 frozen final readout
Purpose: preregister eight paired models and one official subset before training, seal all weights and probes, and compute paired-seed inference. Inputs: fixed18plan and certified actual encoders. Outputs: immutable registration, exposure ledger, final predictions and inferential receipts. Runtime: minutes to hours within original12-hour study clock. Checkpoints: checkpoints/<run_id>/final_convex and source states. Resume: unchanged registration, sources, clock and committed seal. Official pixels remain closed until the seal.

# Notebook17 confirmation definitions
Purpose: predeclare one candidate, calculate training-seed count, seal final weights before opening the official evaluation subset, and analyze paired results. Inputs: screening effects, pinned metadata and completed source-bound model records. Outputs: immutable registration/seal and paired inference. Runtime: under one minute for metadata/statistics; feature scoring is handled by the converged evaluator. Checkpoints: referenced final states under checkpoints/<run_id>/. Exact resume: same registration, sources, seal and subset; invalid hashes raise an error.

In [ ]:
from pathlib import Path
import json, os
if "__file__" not in globals():
    __file__=str(Path(os.environ.get("FMM_JEPA_ROOT","/home/benjamin/workspace/ahr_go/fmm_jeppa"))/"experiments/18_learned_kernel/confirmation.ipynb")
_CONFIRMATION_ROOT=Path(__file__).resolve().parents[2]
_boot={"__file__":str(_CONFIRMATION_ROOT/"fmm_jepa/bootstrap.ipynb")}
for _cell in json.loads((_CONFIRMATION_ROOT/"fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if _cell["cell_type"]=="code" and "ahr-export"in _cell.get("metadata",{}).get("tags",[]):
        exec(compile("".join(_cell["source"]),"bootstrap#ahr-export","exec"),_boot)
_boot["install_notebook_package"](_CONFIRMATION_ROOT)


In [ ]:
"""Frozen single-candidate confirmation; official test stays closed until a valid seal."""
from pathlib import Path
import copy, hashlib, json, math, random, statistics, time
import numpy as np
from scipy import stats
from fmm_jepa.training import atomic_json,digest_file
from fmm_jepa.food101_training import run_writer

def hash_object(value):return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
def read(path):return json.loads(Path(path).read_text())
def immutable(path,value):
    path=Path(path);value=json.loads(json.dumps(value,sort_keys=True,allow_nan=False))
    if path.exists():
        if read(path)!=value:raise ValueError('Frozen confirmation record changed: '+str(path))
    else:atomic_json(path,value)
    return value
def select_closed_ids(bundle,per_class=50,seed=20262017):
    ids=list(bundle['heldout_ids']);groups={};forbidden=set(bundle['train_ids'])|set(bundle['probe_train_ids'])|set(bundle['dev_ids'])
    if forbidden&set(ids):raise ValueError('Heldout membership leaked into training/development')
    for sid in ids:
        if not sid.startswith('food101:test:'):raise ValueError('Only original official-test IDs may enter confirmation')
        groups.setdefault(int(bundle['dataset'].index[sid]['label']),[]).append(sid)
    if len(groups)!=101 or any(len(v)<per_class for v in groups.values()):raise ValueError('Complete balanced official-test class coverage unavailable')
    rng=random.Random(seed);selected=[]
    for label in sorted(groups):selected.extend(sorted(rng.sample(sorted(groups[label]),per_class)))
    return {'ids':selected,'membership_sha256':hash_object(selected),'selection_seed':seed,'per_class':per_class,
      'images':len(selected),'classes':101,'dataset_fingerprint':bundle['dataset'].fingerprint,
      'metadata_only':True,'predictive_pixels_read':0,'status':'CLOSED'}
def normalized_probe_config(root,plan):
    import types
    from fmm_jepa.bootstrap import TaggedNotebookLoader
    path=Path(root)/'experiments/15_vit_geometry/evaluation.ipynb';module=types.ModuleType('fmm18_registered_probe');module.__file__=str(path);TaggedNotebookLoader(path).exec_module(module)
    return module.probe_configuration(plan.get('probe_convex'))
def verify_sources(root,sources):
    root=Path(root).resolve()
    for relative,expected in sources.items():
        path=(root/relative).resolve()
        if not path.is_relative_to(root)or path.suffix!='.ipynb':raise ValueError('Invalidregistereddefinitionpath')
        source='\n'.join(''.join(c['source'])for c in json.loads(path.read_text())['cells']if c['cell_type']=='code'and'ahr-export'in c.get('metadata',{}).get('tags',[]))
        if hashlib.sha256(source.encode()).hexdigest()!=expected:raise ValueError('Registereddefinitionchanged:'+relative)
    return True
def prior_confirmation_seeds(root,artifact=None):
    target=(Path(artifact)/'confirmation_registration.json').resolve()if artifact is not None else None;used=set()
    for parent in ('notebook_16','notebook_17','notebook_18'):
        for path in (Path(root)/'artifacts'/parent).rglob('confirmation_registration.json'):
            if target is not None and path.resolve()==target:continue
            used.update(map(int,read(path).get('seeds',[])))
    return used

def register_confirmation(root,artifact,plan,bundle,configuration=None):
    """One fixed candidate, eight fresh pairs; no development-based promotion."""
    root,artifact=Path(root).resolve(),Path(artifact);cfg=configuration or {}
    seeds=list(plan.get('registered_seeds',plan.get('seeds',[])))
    if seeds!=list(range(20262101,20262109))or len(set(seeds))!=8:raise ValueError('Exactly the eight predeclared fresh18pairs required')
    if prior_confirmation_seeds(root,artifact)&set(seeds):raise ValueError('Previously reserved training seed cannot enter18')
    if plan.get('variants')!=['vit','optional_geometry']:raise ValueError('Exactly ViT and one fixed learned candidate required')
    registry_path=root/'data/food101/final_evaluation_registry.json';registry=read(registry_path)if registry_path.exists()else{'runs':{}}
    path=artifact/'confirmation_registration.json';old=read(path)if path.exists()else None
    if old is not None:heldout=old['closed_holdout']
    else:
        exposed={sid for entry in registry['runs'].values()for sid in entry['ids']}
        heldout=select_closed_ids({**bundle,'heldout_ids':[sid for sid in bundle['heldout_ids']if sid not in exposed]},50,20262118)
    reg={'schema':'notebook18-confirmation-registration/v1','run_id':plan['run_id'],'created_unix':time.time(),
      'plan_hash':hash_object(plan),'source_hashes':plan['source_hashes'],'seeds':seeds,'variants':plan['variants'],
      'closed_holdout':heldout,'probe_configuration':normalized_probe_config(root,plan),'probe_training_ids_sha256':hash_object(list(bundle['probe_train_ids'])),
      'primary_metric':'Balanced official-test top-1 accuracy with converged fixed-L2 linear probe after equal1800active-wall-seconds in full JEPA training.',
      'inferential_unit':'Independent paired training seed; image count is not replicate count','primary_family':['learned_kernel_candidate_minus_vit'],
      'alpha':.05,'minimum_effect_pp':.5,'comparison_budget':plan['active_wall_seconds_per_arm'],'registered_pair_count':8,
      'power_statement':'Eight pairs are a fixed feasibility choice. No new-candidate pilot variance is available; significance is not guaranteed.',
      'success_rule':'All eight paired primary accuracy endpoints complete, finite and certified; mean gain>=0.5pp and paired95%t interval wholly above zero.',
      'secondary':'Paired DEV initial/final gains, exact sign sensitivity, kNN and actual update/throughput differences are descriptive; kNN cutoff remains UNKNOWN and does not invalidate a certified primary accuracy; no selection or rerun from these.',
      'checkpoint_selection':'Last completed update within registered equal active-wall allowance; no DEV or test checkpoint selection',
      'opening_policy':'All trained models and training-only certified probes freeze before opening; positive DEV accuracy or ViT mimicry is not required.',
      'opened_status':'CLOSED','test_pixels_read':0,
      'limitations':['Inference conditional on paired-effect assumptions and fixed5050image subset.','Equal wall time may yield different update counts.','Nonsignificance does not prove equivalence.']}
    if old is not None:reg['created_unix']=old['created_unix']
    return immutable(path,reg)

def verify_gate_evidence(root,gate_receipts):
    root=Path(root).resolve()
    for gate in gate_receipts:
        record=gate.get('evidence')
        if not isinstance(record,dict)or not record.get('path')or not record.get('sha256'):raise ValueError('Gate evidence file and checksum required')
        path=Path(record['path']).resolve()
        if not path.is_relative_to(root/'artifacts')or not path.is_file()or digest_file(path)!=record['sha256']:raise ValueError('Gate evidence file changed or outside scientific artifacts')
    return True

def seal_final_evaluation(root,artifact,registration,models,gate_receipts):
    root,artifact=Path(root),Path(artifact)
    if registration!=read(artifact/'confirmation_registration.json'):raise ValueError('Registered final protocol changed')
    verify_sources(root,registration['source_hashes'])
    if registration.get('opened_status')!='CLOSED':raise ValueError('Initial frozen registration mustremainCLOSED')
    required={(seed,variant)for seed in registration['seeds']for variant in registration['variants']}
    actual={(int(row['seed']),row['variant'])for row in models}
    if actual!=required or len(models)!=len(required):raise ValueError('Every fresh paired model required before final opening')
    for gate in gate_receipts:
        if gate.get('status')!='PASS'or not gate.get('passed',False)or gate.get('fixture_source'):raise ValueError('Required numerical or exact-resume evidence did not pass')
    if not {'operator','exact_resume'}.issubset({g.get('role')for g in gate_receipts}):raise ValueError('Own-operator numerical correctness and actual exact-resume receipts required')
    verify_gate_evidence(root,gate_receipts)
    frozen=[]
    for row in models:
        if row.get('status')!='COMPLETE'or not row.get('fair_wall_budget_verified',False):raise ValueError('Incomplete orunequal-wall trainingcannotopenofficialevaluation')
        rec=row['source_encoder'];path=Path(rec['path']).resolve()
        if not path.is_relative_to(root/'checkpoints'/registration['run_id']):raise ValueError('Finalweightsmustbelongtofreshregisteredrun')
        if digest_file(path)!=rec['sha256']:raise ValueError('Finalcheckpointchecksum mismatch')
        if 'training_probe_record'not in row or 'training_probe_checkpoint'not in row:raise ValueError('Converged training-only classifier must freeze with each encoder')
        for key in ('training_probe_record','training_probe_checkpoint'):
            rec=row[key]
            if digest_file(rec['path'])!=rec['sha256']:raise ValueError('Frozen training probe checksum differs')
        frozen.append(copy.deepcopy(row))
    seal={'schema':'notebook18-final-evaluation-seal/v1','registration_path':str(artifact/'confirmation_registration.json'),
      'registration_sha256':digest_file(artifact/'confirmation_registration.json'),'registration_hash':hash_object(registration),
      'plan_hash':registration['plan_hash'],'source_hashes':registration['source_hashes'],'heldout_membership_sha256':registration['closed_holdout']['membership_sha256'],
      'models':frozen,'gate_receipts':gate_receipts,'all_registered_models_frozen':True,'state':'AUTHORIZED_FINAL_EVALUATION',
      'selection_after_opening':'FORBIDDEN','opened_unix':time.time()}
    p=artifact/'final_evaluation_seal.json'
    if p.exists():seal['opened_unix']=read(p)['opened_unix']
    # Reserve this subset durably before first possible prediction. An interrupted
    # opening remains exposed conservatively, and cannot be reused as fresh data.
    registry_path=root/'data/food101/final_evaluation_registry.json'
    with run_writer(root/'data/food101/final_evaluation_registry_writer'):
        registry=read(registry_path)if registry_path.exists()else{'schema':'food101-final-exposure-registry/v1','runs':{}}
        reg_sha=seal['registration_sha256'];ids=registration['closed_holdout']['ids'];key=registration['run_id']
        for other,entry in registry['runs'].items():
            if other!=key and set(entry['ids'])&set(ids):raise ValueError('Finalsubsetwasalreadyreserved/opened byanotherexperiment')
        if key in registry['runs']:
            entry=registry['runs'][key]
            if entry['registration_sha256']!=reg_sha or entry['ids']!=ids:raise ValueError('Existingfinalexposureidentitychanged')
            seal['opened_unix']=entry['opened_unix']
        else:
            registry['runs'][key]={'registration_sha256':reg_sha,'ids':ids,'opened_unix':seal['opened_unix'],'policy':'Reserved conservatively before prediction; these images are not fresh for later studies.'}
            atomic_json(registry_path,registry)
        return immutable(p,seal)
def verify_seal(root,artifact,seal,requested_ids,model_record=None):
    root,artifact=Path(root),Path(artifact);p=artifact/'final_evaluation_seal.json'
    if not p.exists()or read(p)!=seal or seal.get('state')!='AUTHORIZED_FINAL_EVALUATION':raise PermissionError('Avalidfrozenfinal-evaluationsealisrequired')
    regpath=Path(seal['registration_path'])
    if regpath!=artifact/'confirmation_registration.json'or digest_file(regpath)!=seal['registration_sha256']:raise ValueError('Finalregistrationchecksum mismatch')
    reg=read(regpath);verify_sources(root,reg['source_hashes'])
    verify_gate_evidence(root,seal['gate_receipts'])
    for row in seal['models']:
        for key in ('training_probe_record','training_probe_checkpoint'):
            record=row[key]
            if digest_file(record['path'])!=record['sha256']:raise ValueError('Sealed classifier checksum changed before official pixels')
    if hash_object(reg)!=seal['registration_hash']or hash_object(list(requested_ids))!=seal['heldout_membership_sha256']:raise ValueError('Finalmembership/protocol changed')
    if model_record is not None:
        rec=model_record.get('source_encoder',model_record)
        if not any(row['source_encoder']['path']==rec['path']and row['source_encoder']['sha256']==rec['sha256']for row in seal['models']):raise ValueError('Unsealedmodel')
        if digest_file(rec['path'])!=rec['sha256']:raise ValueError('Frozencheckpointchanged')
    return reg
def analyze_confirmation(registration,rows):
    seeds=registration['seeds'];variants=registration['variants'];effects=[];paired=[]
    for seed in seeds:
        group={r['variant']:r for r in rows if int(r['seed'])==seed}
        if set(group)!=set(variants)or sum(int(r['seed'])==seed for r in rows)!=len(variants):return {'status':'UNKNOWN','reason':'Missingor duplicatepaired endpoints'}
        for r in group.values():
            if r.get('status')!='COMPLETE'or not r.get('converged')or not r.get('numerical_valid',False)or r.get('fixture_source'):return {'status':'UNKNOWN','reason':'Nonconverged/invalid/non-scientific finalendpoint'}
            if not isinstance(r.get('accuracy_percent'),(int,float))or not math.isfinite(float(r['accuracy_percent']))or not 0<=float(r['accuracy_percent'])<=100:return {'status':'UNKNOWN','reason':'Invalid final accuracy'}
            if r.get('heldout_membership_sha256')!=registration['closed_holdout']['membership_sha256']:raise ValueError('Finalpredictionmembershipdiffers')
        delta=float(group['optional_geometry']['accuracy_percent']-group['vit']['accuracy_percent']);effects.append(delta)
        paired.append({'seed':seed,'vit_percent':group['vit']['accuracy_percent'],'candidate_percent':group['optional_geometry']['accuracy_percent'],'delta_pp':delta})
    if len(rows)!=2*len(seeds):raise ValueError('Unregisteredendpointincluded')
    values=np.asarray(effects);n=len(values);mean=float(values.mean());sd=float(values.std(ddof=1));sem=sd/math.sqrt(n)
    if sd==0:
        ci=None;p=None;method='UNDEFINED_ZERO_VARIANCE_PAIRED_T; exactdirectional sensitivityreported separately'
    else:
        ci=list(map(float,stats.t.interval(1-registration['alpha'],n-1,loc=mean,scale=sem)));p=float(stats.ttest_1samp(values,0).pvalue);method='Registeredtwo-sidedpairedt'
    positives=int((values>0).sum());negatives=int((values<0).sum());nonzero=positives+negatives
    sign=float(stats.binomtest(positives,nonzero,.5,alternative='two-sided').pvalue)if nonzero else 1.
    significant=ci is not None and ci[0]>0 and p is not None and p<registration['alpha'];meaningful=mean>=registration['minimum_effect_pp']
    return {'status':'COMPLETE','decision':'UNKNOWN_DEGENERATE_VARIANCE'if ci is None else'SUPPORTS_REGISTERED_ACCURACY_GAIN'if significant and meaningful else'NO_CONFIRMED_MEANINGFUL_ACCURACY_GAIN',
      'n_training_pairs':n,'mean_delta_pp':mean,'raw_ci95_pp':ci,'paired_t_two_sided_p':p,'method':method,
      'minimum_effect_pp':registration['minimum_effect_pp'],'positive_pairs':positives,'negative_pairs':negatives,'ties':n-nonzero,
      'exact_sign_test_two_sided_p':sign,'paired_results':paired,'primary_family_size':1,
      'limitations':['Inference is conditional on the paired-effect distribution and fixed official subset.','Exact sign sensitivity is reported separately; images are not independent training replicates.','Nonsignificance does not establish equivalence or improvement.']}
def export_hash():
    p=Path(__file__);return hashlib.sha256('\n'.join(''.join(c['source'])for c in json.loads(p.read_text())['cells']if c['cell_type']=='code'and'ahr-export'in c.get('metadata',{}).get('tags',[])).encode()).hexdigest()

def bind_sealed_evaluation(root,model_builder,source_hashes):
    """Bind our validator immediately before either official-data verification."""
    from fmm_jepa.bootstrap import TaggedNotebookLoader
    import types
    path=Path(root)/'experiments/15_vit_geometry/evaluation.ipynb'
    module=types.ModuleType('fmm18_sealed_evaluation');module.__file__=str(path);TaggedNotebookLoader(path).exec_module(module)
    module.bind_model_builder(model_builder,source_hashes)
    # The frozen15 evaluator defaults to16's verifier. Bind only this isolated
    # instance to17 so gate evidence SHA checks recur before first test pixels.
    module._seal_verifier=lambda requested_root:verify_seal
    return module

def seal_and_score(root,artifact,plan,bundle,training,evaluation,deadline):
    """Open only the preselected official subset after all paired weights and gates freeze."""
    root,artifact=Path(root),Path(artifact);registration=read(artifact/'confirmation_registration.json')
    if registration['plan_hash']!=hash_object(plan):raise ValueError('Finalplan differsfrom preregistration')
    if training.get('status')!='COMPLETE'or not training.get('fair_wall_budget_verified',False):return {'status':'UNKNOWN','reason':'Incompleteor unverifiedequal-active-walltraining'}
    arms=training.get('arms',{});models=[]
    for arm in plan['arms']:
        if arm['id']not in arms:raise ValueError('Missingfreshtrainedarm')
        saved=arms[arm['id']];source=saved.get('source_encoder',saved.get('latest_checkpoint'))
        if not source:raise ValueError('Finaltrainedarmhasnofrozenencoderrecord')
        prior=[r for r in evaluation.get('rows',[])if int(r['seed'])==int(arm['seed'])and r['variant']==arm['variant']]if isinstance(evaluation,dict)else[]
        if len(prior)!=1 or prior[0].get('status')!='COMPLETE'or not prior[0].get('converged')or not prior[0].get('numerical_valid',False):raise ValueError('Every final model requires one certified training-only probe before sealing')
        probe=prior[0]['probe_receipt'];probe_path=Path(probe['path']).resolve()
        if not probe_path.is_relative_to(root/'checkpoints'/plan['run_id'])or digest_file(probe_path)!=probe['sha256']:raise ValueError('Final training probe receipt changed')
        probe_state=read(probe_path)['latest_checkpoint']
        if digest_file(probe_state['path'])!=probe_state['sha256']:raise ValueError('Final certified probe checkpoint changed')
        models.append({'seed':arm['seed'],'variant':arm['variant'],'arm_id':arm['id'],'status':'COMPLETE',
          'fair_wall_budget_verified':bool(saved.get('fair_wall_budget_verified',training['fair_wall_budget_verified'])),'source_encoder':source,
          'training_probe_record':probe,'training_probe_checkpoint':probe_state})
    receipts=training.get('gate_receipts',plan.get('gate_receipts',[]))
    seal=seal_final_evaluation(root,artifact,registration,models,receipts)
    if callable(evaluation):model_builder=evaluation
    elif isinstance(evaluation,dict)and callable(evaluation.get('model_builder')):model_builder=evaluation['model_builder']
    else:raise ValueError('Final readout requires the frozen17 model builder')
    module=bind_sealed_evaluation(root,model_builder,plan['source_hashes'])
    rows=[];heldout=registration['closed_holdout']['ids']
    with run_writer(artifact/'final_evaluation_writer'):
        for arm,model in zip(plan['arms'],models):
            if time.time()>=deadline:return {'status':'UNKNOWN','reason':'Finalreadoutcutoff','completed_endpoints':len(rows),'official_test':'OPENED_FOR_FROZEN_PROTOCOL_ONLY'}
            score_config=copy.deepcopy(arm['config']);score_config['probe_convex']=registration['probe_configuration']
            from fmm_jepa.bootstrap import TaggedNotebookLoader
            import types
            cp=root/'experiments/18_learned_kernel/common.ipynb';common=types.ModuleType('fmm18_final_environment');common.__file__=str(cp);TaggedNotebookLoader(cp).exec_module(common)
            common.environment(score_config);common.pipeline(root).install_input_pipeline(int(plan['input_workers']),root)
            candidates=evaluation.get('rows',[])if isinstance(evaluation,dict)else[]
            prior=[r for r in candidates if int(r['seed'])==int(arm['seed'])and r['variant']==arm['variant']]
            if len(prior)>1:raise ValueError('Duplicate converged training probe for final model')
            probe_record=prior[0]['probe_receipt']if prior and prior[0].get('converged')else None
            record=module.evaluate_model_on_ids(root,artifact,score_config,model['source_encoder'],bundle['probe_train_ids'],heldout,bundle['dataset'],
              training_probe_record=probe_record,deadline_unix=deadline,seal_record=seal)
            if record.get('status')=='UNKNOWN_KNN_CUTOFF'and record.get('converged')and record.get('numerical_valid',False):
                record.update(status='COMPLETE',primary_readout_status='COMPLETE',secondary_knn_status='UNKNOWN_KNN_CUTOFF')
            record.update(seed=arm['seed'],variant=arm['variant'],heldout_membership_sha256=registration['closed_holdout']['membership_sha256'])
            rows.append(record);atomic_json(artifact/'final_evaluation_progress.json',{'status':'RUNNING','completed_endpoints':len(rows),'total_endpoints':len(plan['arms']),'rows':rows})
            if record.get('status')!='COMPLETE'or not record.get('converged'):return {'status':'UNKNOWN','reason':'Final endpoint incomplete or classifier not converged','completed_endpoints':len(rows),'rows':rows}
        result=analyze_confirmation(registration,rows);result.update(official_test='EVALUATED_ON_FROZEN5050_SUBSET',final_evaluation_seal_sha256=digest_file(artifact/'final_evaluation_seal.json'),rows=rows)
        atomic_json(artifact/'confirmation_results.json',result);return result


In [ ]:
def analyze_development(plan,evaluation):
    """Secondary own-initial/final DEV results; never selects official opening."""
    expected={(int(s),v)for s in plan['seeds']for v in plan['variants']}
    maps=[]
    for group in(evaluation.get('initial_rows',[]),evaluation.get('rows',[])):
        if len(group)!=len(expected)or {(int(r['seed']),r['variant'])for r in group}!=expected:return {'status':'UNKNOWN','reason':'Missing or duplicate initial/final DEV endpoint'}
        if any(r.get('status')!='COMPLETE'or not r.get('converged')or not r.get('numerical_valid',False)or r.get('fixture_source')for r in group):return {'status':'UNKNOWN','reason':'Uncertified DEV endpoint'}
        if any(not math.isfinite(float(r.get('accuracy_percent',float('nan'))))or (r.get('knn_accuracy_percent')is not None and not math.isfinite(float(r['knn_accuracy_percent'])))for r in group):return {'status':'UNKNOWN','reason':'Nonfinite DEV metric'}
        maps.append({(int(r['seed']),r['variant']):r for r in group})
    start,end=maps;pairs=[]
    for seed in plan['seeds']:
        v=end[(seed,'vit')];c=end[(seed,'optional_geometry')];v0=start[(seed,'vit')];c0=start[(seed,'optional_geometry')]
        pairs.append({'seed':seed,'vit_percent':v['accuracy_percent'],'candidate_percent':c['accuracy_percent'],'delta_pp':c['accuracy_percent']-v['accuracy_percent'],
            'vit_initial_percent':v0['accuracy_percent'],'candidate_initial_percent':c0['accuracy_percent'],
            'vit_gain_from_initial_pp':v['accuracy_percent']-v0['accuracy_percent'],'candidate_gain_from_initial_pp':c['accuracy_percent']-c0['accuracy_percent'],
            'knn_delta_pp':None if c.get('knn_accuracy_percent')is None or v.get('knn_accuracy_percent')is None else c['knn_accuracy_percent']-v['knn_accuracy_percent']})
    delta=np.array([p['delta_pp']for p in pairs],dtype=float);mean=float(delta.mean());sd=float(delta.std(ddof=1));ci=None if sd==0 else list(map(float,stats.t.interval(.95,len(pairs)-1,loc=mean,scale=sd/math.sqrt(len(pairs)))))
    return {'status':'COMPLETE','paired_results':pairs,'mean_delta_pp':mean,'raw_ci95_pp':ci,'n_training_pairs':len(pairs),'positive_pairs':int((delta>0).sum()),
        'vit_gain_from_initial_pp':float(np.mean([p['vit_gain_from_initial_pp']for p in pairs])),
        'candidate_gain_from_initial_pp':float(np.mean([p['candidate_gain_from_initial_pp']for p in pairs])),
        'scope':'Secondary reused-development description; no architecture,checkpoint,subset or opening selected from these results','official_test':'CLOSED'}
